

**Assignment:** You are provided with the dataset `Student_Performance.csv`.

In this notebook, we will perform basic data preprocessing:

1. Load the dataset  
2. Check dataset information  
3. Handle missing values  
4. Remove duplicate rows  
5. Clean text columns  
6. Encode categorical columns  
7. Scale numeric columns  
8. Save the cleaned dataset as `Student_Performance_Cleaned.csv`  
9. Optional: find top students and average marks by gender

**Dataset link:** Kaggle — Students Performance Dataset  
https://www.kaggle.com/datasets/rabieelkharoua/students-performance-dataset

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

In [1]:
df = pd.read_csv('Student_Performance.csv')

NameError: name 'pd' is not defined

## Step 3: Check First and Last Rows

`head()` shows the first 5 rows.  
`tail()` shows the last 5 rows.

This helps us understand what the dataset looks like.

In [ ]:
print("First 5 rows:")
display(df.head())

print("Last 5 rows:")
display(df.tail())

## Step 4: Check Dataset Shape

`shape` tells us how many rows and columns are in the dataset.

- Rows = records/students
- Columns = features/information

In [ ]:
print("Rows and Columns:", df.shape)
print("Total Rows:", df.shape[0])
print("Total Columns:", df.shape[1])

## Step 5: Check Column Names

This shows all column names available in the dataset.

In [ ]:
print("Column Names:")
print(df.columns.tolist())

## Step 6: Basic Dataset Information

`info()` tells us:

- Column names
- Non-null values
- Data types

This is very useful before cleaning the data.

In [ ]:
df.info()

## Step 7: Check Missing Values

Missing values mean empty/null values in the dataset.

We use `isnull().sum()` to check how many missing values are present in each column.

In [ ]:
print("Missing values in each column:")
print(df.isnull().sum())

## Step 8: Handle Missing Values

Simple beginner rule:

- Numeric columns: fill missing values with mean
- Text/categorical columns: fill missing values with mode

Mean = average value  
Mode = most repeated value

In [ ]:
# Separate numeric and categorical columns
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = df.select_dtypes(include=['object']).columns

print("Numeric Columns:", list(numeric_cols))
print("Categorical Columns:", list(categorical_cols))

# Fill missing numeric values with mean
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].mean())

# Fill missing categorical values with mode
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values after handling:")
print(df.isnull().sum())

## Step 9: Remove Duplicate Rows

Duplicate rows are repeated records.

We first check how many duplicates exist, then remove them.

In [ ]:
print("Duplicate rows before removing:", df.duplicated().sum())

df = df.drop_duplicates()

print("Duplicate rows after removing:", df.duplicated().sum())
print("New shape:", df.shape)

## Step 10: Clean Column Names

Sometimes column names have spaces or capital letters.

We clean them by:

- Converting to lowercase
- Replacing spaces with underscores

Example: `Hours Studied` becomes `hours_studied`

In [ ]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

print("Cleaned Column Names:")
print(df.columns.tolist())

## Step 11: Clean Text Columns

Text values can contain extra spaces or inconsistent capital letters.

Example:

- ` male ` becomes `Male`
- `female` becomes `Female`

This step only works on text columns.

In [ ]:
text_cols = df.select_dtypes(include=['object']).columns

for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.title()

print("Text columns cleaned:", list(text_cols))
display(df.head())

## Step 12: Check Unique Values in Text Columns

This helps us see different categories in each text column.

Example: Male/Female, Yes/No, Grade A/B/C, etc.

In [ ]:
for col in text_cols:
    print("Column:", col)
    print(df[col].unique())
    print("-" * 40)

## Step 13: Encode Categorical Columns

Machine learning models do not understand text directly.

So we convert text into numbers.

Here we use **Label Encoding** for beginner level.

Example:

- Yes = 1
- No = 0

In [ ]:
label_encoders = {}

for col in text_cols:
    le = LabelEncoder()
    df[col + "_encoded"] = le.fit_transform(df[col])
    label_encoders[col] = le

print("Categorical columns encoded successfully.")
display(df.head())

## Step 14: One-Hot Encoding Alternative

One-Hot Encoding creates separate columns for each category.

Example:

`gender` becomes:

- gender_Male
- gender_Female

We create a separate dataframe `df_onehot` so original encoded dataframe remains safe.

In [ ]:
df_onehot = pd.get_dummies(df, columns=list(text_cols), drop_first=False)

print("One-hot encoded dataset shape:", df_onehot.shape)
display(df_onehot.head())

## Step 15: Scale Numeric Features

Scaling means converting numeric values to a similar range.

Here we use **Min-Max Scaling**, which converts values between 0 and 1.

Formula:

`scaled_value = (value - min) / (max - min)`

In [ ]:
# Select numeric columns again after encoding
numeric_cols_after = df.select_dtypes(include=['int64', 'float64']).columns

scaler = MinMaxScaler()

df_scaled = df.copy()
df_scaled[numeric_cols_after] = scaler.fit_transform(df_scaled[numeric_cols_after])

print("Numeric columns scaled between 0 and 1.")
display(df_scaled.head())

## Step 16: Final Cleaned Dataset

Now our dataset is cleaned, encoded, and scaled.

We will save it as a CSV file.

In [ ]:
df_scaled.to_csv("Student_Performance_Cleaned.csv", index=False)

print("Cleaned dataset saved successfully as Student_Performance_Cleaned.csv")

## Optional Task 1: Find Top Students by Marks/Performance

Different datasets may have different column names.

This code tries to find a column related to performance or marks, then displays top 5 records.

In [ ]:
possible_score_cols = [
    "performance_index", "marks", "score", "exam_score", "final_score", "math_score", "writing_score", "reading_score"
]

score_col = None
for col in possible_score_cols:
    if col in df.columns:
        score_col = col
        break

if score_col:
    print("Score column found:", score_col)
    top_students = df.sort_values(by=score_col, ascending=False).head(5)
    display(top_students)
else:
    print("No marks/performance column found. Please check your column names:")
    print(df.columns.tolist())

## Optional Task 2: Average Marks by Gender

This code checks if a gender column exists.

If gender and score columns are available, it calculates average score by gender.

In [ ]:
possible_gender_cols = ["gender", "sex"]

gender_col = None
for col in possible_gender_cols:
    if col in df.columns:
        gender_col = col
        break

if gender_col and score_col:
    print("Average", score_col, "by", gender_col)
    display(df.groupby(gender_col)[score_col].mean())
else:
    print("Gender column or score column not found.")
    print("Available columns:", df.columns.tolist())

## Final Summary

In this notebook, we completed these preprocessing steps:

1. Loaded dataset  
2. Checked rows, columns, and missing values  
3. Filled missing numeric values with mean  
4. Filled missing categorical values with mode  
5. Removed duplicate rows  
6. Cleaned column names  
7. Cleaned text values  
8. Encoded categorical values  
9. Scaled numeric features  
10. Saved final cleaned dataset  

This is the basic preprocessing workflow before machine learning.